In [0]:
customer_landing_path = (
    "abfss://landing@batchstorage11.dfs.core.windows.net/dbo.customers.txt"
)

customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ",")
    .csv(customer_landing_path)
)


In [0]:
bronze_customer_path = (
    "abfss://bronze@batchstorage11.dfs.core.windows.net/customers"
)

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_customer_path)

print("Customer data successfully written to Bronze!")
print("Customer record count:", customers_df.count())

Customer data successfully written to Bronze!
Customer record count: 5


In [0]:
from pyspark.sql.functions import col, trim, lower

bronze_customer_path = (
    "abfss://bronze@batchstorage11.dfs.core.windows.net/customers"
)

silver_customer_path = (
    "abfss://silver@batchstorage11.dfs.core.windows.net/customers"
)

# Read customer data from Bronze
bronze_customers_df = (
    spark.read.format("delta").load(bronze_customer_path)
)

# Clean and validate customer data
silver_customers_df = (
    bronze_customers_df
    .dropDuplicates(["customer_id"])
    .filter(
        col("customer_id").isNotNull()
        & col("customer_name").isNotNull()
    )
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("email", lower(trim(col("email"))))
)

# Write cleaned data to Silver
silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_customer_path)

print("Customer Silver transformation completed!")
print("Silver record count:", silver_customers_df.count())

Customer Silver transformation completed!
Silver record count: 5


In [0]:
from pyspark.sql.functions import count

silver_customer_path = (
    "abfss://silver@batchstorage11.dfs.core.windows.net/customers"
)

gold_customer_path = (
    "abfss://gold@batchstorage11.dfs.core.windows.net/customer_summary"
)

# Read cleaned customer data from Silver
silver_customers_df = (
    spark.read.format("delta").load(silver_customer_path)
)

# Count customers by city
gold_customers_df = (
    silver_customers_df
    .groupBy("city")
    .agg(count("customer_id").alias("total_customers"))
)

# Write summary to Gold
gold_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_customer_path)

display(gold_customers_df)

city,total_customers
Chennai,1
Mumbai,1
Pune,1
Bengaluru,1
Hyderabad,1


In [0]:
gold_customer_path = (
    "abfss://gold@batchstorage11.dfs.core.windows.net/customer_summary"
)

gold_customers_df = (
    spark.read.format("delta").load(gold_customer_path)
)

display(gold_customers_df)

print("Gold customer summary validation completed!")
print("Total cities:", gold_customers_df.count())

city,total_customers
Chennai,1
Mumbai,1
Pune,1
Bengaluru,1
Hyderabad,1


Gold customer summary validation completed!
Total cities: 5


## Customer Data Pipeline – Execution Completed

The customer data pipeline has completed the following stages:

- **Landing:** Read customer data from `dbo.customers.txt`.
- **Bronze:** Ingested the source data into Delta format.
- **Silver:** Removed duplicate customer IDs, validated required fields, trimmed names and cities, and normalized email addresses.
- **Gold:** Generated a city-level summary showing the number of customers per city.
- **Validation:** Read the Gold output and verified the summary.

**Output locations:**
- Bronze: `abfss://bronze@batchstorage11.dfs.core.windows.net/customers`
- Silver: `abfss://silver@batchstorage11.dfs.core.windows.net/customers`
- Gold: `abfss://gold@batchstorage11.dfs.core.windows.net/customer_summary`

**Status:** Customer data processing completed.